# MTSD - Local Object Detection (RF-DETR)

---

**Name:** Andrea Filiberto Lucas
**Project:** MTSD-Supervised-Detection

---

RF-DETR model-scale benchmark with Weights & Biases tracking on the **Maltese
Traffic Sign Dataset (MTSD)** — 12 QA-verified sign classes prepared by
`Prepare-MTSD-Detection-Dataset.ipynb` into `Datasets/MTSD/Prepared/MTSD-COCO/`.
Trains RF-DETR-N, RF-DETR-S, and RF-DETR-M under the shared protocol.

This notebook mirrors the MDWD supervised benchmark template (same shared training
protocol, logging, and model-scale ablation structure) with MTSD-specific inputs and
outputs, so results are directly comparable across the two dissertation tracks:

- runs -> `Results/MTSD-Runs/RF-DETR-MTSD/`
- consolidated exports -> `Results/MTSD-Results/RF-DETR/Model-Size-Comparison/`
- W&B project -> `MTSD-Supervised-Detection`

> **Secrets.** W&B credentials are read from the repository-root `.env` (git-ignored) or
> an existing `wandb login`; API keys are never hardcoded, printed, or committed. Keep it
> that way even though this repository is private.

> **Training cell — intentionally left unexecuted during repository setup.** The final
> cell ships with `RUN_TRAINING = False`; flip it to `True` only when training is
> actually intended. Existing run directories are never overwritten.


In [ ]:
import os
from pathlib import Path

# Load local secrets from the repository-root .env file. The file is ignored by Git.
for candidate in [p / ".env" for p in [Path.cwd(), *Path.cwd().parents]]:
    if candidate.exists():
        for line in candidate.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip("\"'"))
        break

print("W&B credentials loaded successfully!" if os.getenv("WANDB_API_KEY") else "WANDB_API_KEY is not set; using existing W&B login (if available).")


In [ ]:
import os
import re
import sys
import json
import time
import shutil
import random
import platform
import subprocess
import importlib.util
import importlib.metadata as importlib_metadata
from copy import deepcopy
from datetime import datetime
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

COLORS = {"green": "\033[92m", "red": "\033[91m", "yellow": "\033[93m", "reset": "\033[0m"}


def find_repo_root(start=None):
    start_path = Path(start or os.getcwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "Datasets").exists() and (candidate / "Scripts").exists() and (candidate / "Models").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing Datasets/, Scripts/ and Models/.")


ROOT_DIR = find_repo_root()
PREPARED_DIR = ROOT_DIR / "Datasets" / "MTSD" / "Prepared"
MODELS_DIR = ROOT_DIR / "Models"
DATASET_DIR = PREPARED_DIR / "MTSD-COCO"
COCO_ANNOTATION_NAME = "_annotations.coco.json"

PACKAGE_RULES = {
    "rfdetr": {"import": "rfdetr", "spec": "rfdetr[metrics]"},
    "torch": {"import": "torch"},
    "numpy": {"import": "numpy"},
    "pandas": {"import": "pandas"},
    "wandb": {"import": "wandb"},
    "supervision": {"import": "supervision"},
}


def parse_version(value):
    parts = re.findall(r"\d+", str(value))[:3]
    return tuple(int(part) for part in parts + ["0"] * (3 - len(parts)))


def package_version(package_name):
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None


def version_satisfies(version, rule):
    if version is None:
        return False
    parsed = parse_version(version)
    min_version = rule.get("min")
    max_exclusive = rule.get("max_exclusive")
    return not ((min_version and parsed < parse_version(min_version)) or (max_exclusive and parsed >= parse_version(max_exclusive)))


def install_spec(package_name, rule):
    spec = rule.get("spec", package_name)
    if rule.get("min") and ">=" not in spec:
        spec += f">={rule['min']}"
    if rule.get("max_exclusive") and "<" not in spec:
        spec += f"<{rule['max_exclusive']}"
    return spec


def check_and_install_libraries():
    changed = False
    for package_name, rule in PACKAGE_RULES.items():
        import_name = rule.get("import", package_name)
        installed = importlib.util.find_spec(import_name) is not None
        current_version = package_version(package_name)
        if installed and version_satisfies(current_version, rule):
            version_text = f" ({current_version})" if current_version else ""
            print(f"[{COLORS['green']}OK{COLORS['reset']}] Library '{package_name}' is installed{version_text}.")
            continue
        spec = install_spec(package_name, rule)
        print(f"[{COLORS['yellow']}...{COLORS['reset']}] Installing/updating '{spec}'...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", spec])
        changed = True

    if changed:
        raise RuntimeError("Packages were installed or updated. Restart the Jupyter kernel, then rerun from the top.")


check_and_install_libraries()

import torch
import pandas as pd  # type: ignore
import numpy as np  # type: ignore
from rfdetr import RFDETRNano, RFDETRSmall, RFDETRMedium  # type: ignore


In [ ]:
PREP_NOTEBOOK = ROOT_DIR / "Scripts" / "MTSD-Scripts" / "MTSD-SupervisedNotebooks" / "Prepare-MTSD-Detection-Dataset.ipynb"
PREP_MANIFEST_PATH = PREPARED_DIR / "prep_manifest.json"

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Prepared MTSD COCO dataset not found at {DATASET_DIR}.\n"
        f"Run {PREP_NOTEBOOK.name} first - it builds MTSD-COCO (and MTSD-YOLO) "
        f"from the QA-verified annotations."
    )

PREP_MANIFEST = json.loads(PREP_MANIFEST_PATH.read_text(encoding="utf-8")) if PREP_MANIFEST_PATH.exists() else {}
DATASET_VERSION_TAG = PREP_MANIFEST.get("dataset_version", "mtsdv1")

DATASET_SOURCE = {
    "name": "MTSD",
    "version": DATASET_VERSION_TAG,
    "layout": "coco",
    "prepared_by": "Prepare-MTSD-Detection-Dataset.ipynb",
    "annotations": "QA-verified Final-QA exports (GRP-1..GRP-3)",
    "manifest": str(PREP_MANIFEST_PATH),
    "split_mode": PREP_MANIFEST.get("configuration", {}).get("split_mode"),
    "split_seed": PREP_MANIFEST.get("configuration", {}).get("random_seed"),
    "dataset_augmentation": (
        "offline photometric augmentation (train split)"
        if PREP_MANIFEST.get("configuration", {}).get("offline_augmentation")
        else "no offline augmentation; RF-DETR online augmentation left at package defaults"
    ),
}

missing_annotations = []
for split in ["train", "valid", "test"]:
    annotation_path = DATASET_DIR / split / COCO_ANNOTATION_NAME
    if not annotation_path.exists():
        missing_annotations.append(str(annotation_path))

if missing_annotations:
    raise FileNotFoundError("MTSD COCO annotations not found:\n" + "\n".join(missing_annotations))

def normalize_coco_categories_for_rfdetr(dataset_dir, default_supercategory="mtsd"):
    """Make COCO exports safe for RF-DETR 1.3.x.

    The preparation notebook already writes contiguous 0-based category ids with a
    supercategory, so this is normally a no-op; it is kept as a guard for datasets
    prepared by other tools (RF-DETR loaders use raw COCO category_id values as
    training labels, so 1-based ids can trigger CUDA device-side asserts).
    """
    patched_files = []
    for split in ["train", "valid", "test"]:
        annotation_path = Path(dataset_dir) / split / COCO_ANNOTATION_NAME
        data = json.loads(annotation_path.read_text(encoding="utf-8"))
        changed = False
        categories = sorted(data.get("categories", []) or [], key=lambda item: item["id"])
        id_map = {category["id"]: index for index, category in enumerate(categories)}
        for category in categories:
            if "supercategory" not in category or str(category.get("supercategory")).strip().lower() in ["", "none", "null"]:
                category["supercategory"] = default_supercategory
                changed = True
            new_id = id_map[category["id"]]
            if category["id"] != new_id:
                category["id"] = new_id
                changed = True
        for annotation in data.get("annotations", []) or []:
            category_id = annotation.get("category_id")
            if category_id in id_map and annotation["category_id"] != id_map[category_id]:
                annotation["category_id"] = id_map[category_id]
                changed = True
        data["categories"] = categories
        if changed:
            backup_path = annotation_path.with_suffix(annotation_path.suffix + ".bak")
            if not backup_path.exists():
                shutil.copy2(annotation_path, backup_path)
            annotation_path.write_text(json.dumps(data, indent=2), encoding="utf-8")
            patched_files.append(str(annotation_path))
    return patched_files

patched_annotations = normalize_coco_categories_for_rfdetr(DATASET_DIR)
if patched_annotations:
    print("Patched COCO categories for RF-DETR compatibility:")
    for path in patched_annotations:
        print(" -", path)
else:
    print("COCO categories already RF-DETR compatible.")

print(f"[{COLORS['green']}OK{COLORS['reset']}] MTSD COCO dataset is ready: {DATASET_DIR}")


## Hardware, Reproducibility, and Paths


In [ ]:
RUNS_DIR = ROOT_DIR / "Results" / "MTSD-Runs" / "RF-DETR-MTSD"
RESULTS_DIR = ROOT_DIR / "Results" / "MTSD-Results" / "RF-DETR"
BENCHMARK_RESULTS_DIR = RESULTS_DIR / "Model-Size-Comparison"
RUN_CONFIG_DIR = BENCHMARK_RESULTS_DIR / "run_configs"

for directory in [RUNS_DIR, BENCHMARK_RESULTS_DIR, RUN_CONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def determine_device(prefer_mps=True):
    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
    elif prefer_mps and torch.backends.mps.is_available():
        device = "mps"
        print("Using MPS (Apple Metal Performance Shaders).")
    else:
        device = "cpu"
        print("Defaulting to CPU. Training may be slow without GPU acceleration.")
    return device


def get_hardware_info(device=None):
    device = device or determine_device(prefer_mps=True)
    info = {
        "device": device,
        "platform": platform.platform(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "python": sys.version.replace("\n", " "),
        "torch_version": getattr(torch, "__version__", None),
        "cuda_available": torch.cuda.is_available(),
        "mps_available": torch.backends.mps.is_available(),
        "mps_built": torch.backends.mps.is_built(),
    }
    if torch.cuda.is_available():
        info["cuda_version"] = torch.version.cuda
        info["cuda_device_count"] = torch.cuda.device_count()
        info["cuda_devices"] = []
        for idx in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(idx)
            info["cuda_devices"].append({
                "index": idx,
                "name": props.name,
                "total_memory_gb": round(props.total_memory / (1024 ** 3), 2),
                "major": props.major,
                "minor": props.minor,
                "multi_processor_count": props.multi_processor_count,
            })
    return info


def default_workers(device):
    if platform.system().lower() == "windows":
        return 0
    if device in {"mps", "cpu"}:
        return 0
    return min(8, max(2, (os.cpu_count() or 4) // 2))


def set_global_seed(seed=42, deterministic=True):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        try:
            torch.use_deterministic_algorithms(True, warn_only=True)
        except Exception as exc:
            print(f"Could not enable full deterministic algorithms: {exc}")
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


DEVICE = determine_device(prefer_mps=True)
WORKERS = default_workers(DEVICE)
HARDWARE_INFO = get_hardware_info(DEVICE)
set_global_seed(42, deterministic=True)

hardware_config_path = RUN_CONFIG_DIR / "hardware_and_runtime_config.json"
hardware_config_path.write_text(json.dumps({"hardware": HARDWARE_INFO, "device": DEVICE, "workers": WORKERS}, indent=2, default=str), encoding="utf-8")
print("Saved hardware/runtime config to:", hardware_config_path)


## Weights & Biases


In [ ]:
WANDB_PROJECT = "MTSD-Supervised-Detection"
WANDB_ENTITY = os.getenv("WANDB_ENTITY", "mark-bugeja-university-of-malta")
WANDB_DEFAULT_MODE = os.getenv("WANDB_MODE", "online")
WANDB_AVAILABLE = importlib.util.find_spec("wandb") is not None

if WANDB_AVAILABLE:
    import wandb  # type: ignore
else:
    wandb = None  # type: ignore
    print("W&B is not installed. RF-DETR training can still run with W&B disabled.")


def prepare_wandb(config):
    if not WANDB_AVAILABLE or wandb is None:
        print("W&B unavailable; RF-DETR will train with wandb=False.")
        return False

    requested_mode = str(config.get("wandb", {}).get("mode") or WANDB_DEFAULT_MODE).lower()
    if requested_mode == "disabled":
        os.environ["WANDB_MODE"] = "disabled"
        return False

    os.environ["WANDB_PROJECT"] = config.get("wandb", {}).get("project", WANDB_PROJECT)
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY
    os.environ["WANDB_MODE"] = requested_mode
    os.environ["WANDB_RUN_GROUP"] = config.get("wandb_group", EXPERIMENT_FAMILY)

    try:
        if requested_mode == "online" and os.getenv("WANDB_API_KEY"):
            wandb.login(key=os.getenv("WANDB_API_KEY"), relogin=False, verify=True)
        elif requested_mode == "online":
            print("WANDB_MODE=online but WANDB_API_KEY is not set. Using existing W&B login if available.")
        return True
    except Exception as exc:
        print(f"W&B login failed ({exc}). RF-DETR will continue with WANDB_MODE=offline.")
        os.environ["WANDB_MODE"] = "offline"
        return True


def finish_wandb_run():
    if WANDB_AVAILABLE and wandb is not None and wandb.run is not None:
        try:
            wandb.finish()
        except Exception as exc:
            print(f"Could not finish W&B run cleanly: {exc}")


## Shared Training Template Configuration

The protocol (epochs, image size, batch, optimiser, learning-rate schedule, and disabled
online augmentation) is kept **identical to the MDWD benchmarks** so model-scale effects
are comparable across the two datasets. Note the difference in what that implies: the
MDWD export was pre-augmented ~10x offline by Roboflow, whereas the prepared MTSD dataset
is un-augmented by default - if more training variety is needed, generate offline
photometric augmentations in `Prepare-MTSD-Detection-Dataset.ipynb` (§7) rather than
re-enabling online augmentation here (RF-DETR keeps its package-default pipeline).


In [ ]:
RFDETR_MODEL_REGISTRY = {
    "rfdetr-n": {"class": RFDETRNano, "display_name": "RF-DETR-Nano"},
    "rfdetr-s": {"class": RFDETRSmall, "display_name": "RF-DETR-Small"},
    "rfdetr-m": {"class": RFDETRMedium, "display_name": "RF-DETR-Medium"},
}

# Template control point: edit this list to train any set/order of RF-DETR model scales
# with the exact same dataset, hyperparameters, optimizer, and logging settings.
RFDETR_MODEL_VARIANTS = ["rfdetr-n", "rfdetr-s", "rfdetr-m"]
EXPERIMENT_PREFIX_TEMPLATE = "E{index:03d}"
EXPERIMENT_FAMILY = "RF-DETR-MTSD-Model-Scale-Template"

base_training_config = {
    "experiment_number": 1,
    "experiment_type": "multi-model-template",
    "configuration_source": "shared_rfdetr_mtsdv1_coco_img640_b32_e100_adamw_lr0p001",
    "dataset_path": str(DATASET_DIR),
    "dataset_dir": str(DATASET_DIR),
    "dataset_format": "COCO/MMDetection",
    "dataset_source": DATASET_SOURCE,
    "epochs": 100,
    "imgsz": 640,
    "batch": 32,
    "batch_size": 32,
    "grad_accum_steps": 1,
    "effective_batch_size": 32,
    "optimizer": "AdamW",
    "lr0": 0.001,
    "lr": 0.001,
    "lr_encoder": 0.001,
    "lrf": 0.01,
    "lr_drop": 100,
    "resolution": 640,
    "multi_scale": False,
    "expanded_scales": False,
    "do_random_resize_via_padding": False,
    "weight_decay": 0.0005,
    "momentum": 0.937,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "use_ema": True,
    "gradient_checkpointing": True,
    "checkpoint_interval": 10,
    "early_stopping": True,
    "patience": 10,
    "early_stopping_patience": 10,
    "early_stopping_min_delta": 0.001,
    "seed": 42,
    "deterministic": True,
    "device": DEVICE,
    "workers": WORKERS,
    "project_dir": str(RUNS_DIR),
    "pretrained": True,
    "wandb_group": EXPERIMENT_FAMILY,
    "tags": ["rf-detr", "rfdetr", "mtsd", "multi-model-template", "model-scale-benchmark", "coco", "adamw", "img640", "b32", "lr0p001", "mtsdv1"],
    "notes": "MTSD template notebook: train RF-DETR Nano, Small, and Medium on the prepared MTSD detection dataset with the same shared training protocol as the MTSD YOLO12/YOLO26 notebooks; only the model class changes.",
    "wandb": {"project": WANDB_PROJECT, "mode": WANDB_DEFAULT_MODE},
    "hardware": {
        "device": DEVICE,
        "workers": WORKERS,
        "cuda_available": HARDWARE_INFO.get("cuda_available"),
        "gpu": HARDWARE_INFO.get("cuda_devices", [{}])[0].get("name") if HARDWARE_INFO.get("cuda_devices") else None,
        "gpu_vram_gb": HARDWARE_INFO.get("cuda_devices", [{}])[0].get("total_memory_gb") if HARDWARE_INFO.get("cuda_devices") else None,
    },
}


def safe_run_name(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(text)).strip("-")


base_training_config["created_at"] = datetime.now().isoformat(timespec="seconds")
template_config_path = RUN_CONFIG_DIR / "base_training_template.json"
template_config_path.write_text(json.dumps(base_training_config, indent=2, default=str), encoding="utf-8")

print("Shared training template saved to:", template_config_path)
print("Model variants to train:", RFDETR_MODEL_VARIANTS)
print(json.dumps(base_training_config, indent=2, default=str))


## Training Helpers


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def count_images(split_dir):
    split_dir = Path(split_dir)
    if not split_dir.exists():
        return 0
    return sum(1 for path in split_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTENSIONS)


def load_coco_summary(annotation_path):
    annotation_path = Path(annotation_path)
    data = json.loads(annotation_path.read_text(encoding="utf-8"))
    categories = data.get("categories", []) or []
    annotations = data.get("annotations", []) or []
    images = data.get("images", []) or []
    return {
        "annotation_file": str(annotation_path),
        "images": len(images),
        "annotations": len(annotations),
        "categories": categories,
        "class_names": [category.get("name") for category in categories],
        "class_count": len(categories),
    }


def dataset_metadata(config):
    dataset_dir = Path(config["dataset_dir"])
    metadata = {"dataset_path": config["dataset_path"], "dataset_dir": config["dataset_dir"], **config["dataset_source"]}
    for split in ["train", "valid", "test"]:
        split_dir = dataset_dir / split
        annotation_path = split_dir / COCO_ANNOTATION_NAME
        split_summary = load_coco_summary(annotation_path) if annotation_path.exists() else {"warning": "annotation file missing"}
        split_summary["image_files"] = count_images(split_dir)
        metadata[split] = split_summary
    train_categories = metadata.get("train", {}).get("categories", [])
    metadata["class_names"] = [category.get("name") for category in train_categories]
    metadata["class_count"] = len(train_categories)
    return metadata


def checkpoint_summary(output_dir):
    output_dir = Path(output_dir)
    checkpoints = []
    for path in sorted(output_dir.glob("checkpoint*.pth")):
        checkpoints.append({"name": path.name, "path": str(path), "size_mb": round(path.stat().st_size / (1024 ** 2), 2)})
    return checkpoints


def flatten_metrics(prefix, value, payload):
    if isinstance(value, dict):
        for key, nested in value.items():
            flatten_metrics(f"{prefix}/{key}" if prefix else str(key), nested, payload)
    elif isinstance(value, (int, float, np.number)) and not pd.isna(value):
        payload[prefix] = float(value)


def extract_train_result_metrics(train_result):
    payload = {}
    if isinstance(train_result, dict):
        flatten_metrics("result", train_result, payload)
    else:
        for attr in ["metrics", "results", "summary"]:
            if hasattr(train_result, attr):
                flatten_metrics(attr, getattr(train_result, attr), payload)
    return payload


def log_dataset_metadata_to_wandb(config):
    if wandb is None or wandb.run is None:
        return
    try:
        artifact = wandb.Artifact(
            name=f"mtsd-prepared-{config['dataset_source']['version']}-coco",
            type="dataset",
            metadata=dataset_metadata(config),
        )
        for split in ["train", "valid", "test"]:
            annotation_path = Path(config["dataset_dir"]) / split / COCO_ANNOTATION_NAME
            if annotation_path.exists():
                artifact.add_file(str(annotation_path), name=f"{split}_{COCO_ANNOTATION_NAME}")
        wandb.run.log_artifact(artifact)
    except Exception as exc:
        print(f"Could not log dataset metadata artifact: {exc}")


def log_artifacts_to_wandb(config, output_dir):
    if wandb is None or wandb.run is None:
        return
    output_dir = Path(output_dir)
    try:
        for pattern in ["*.csv", "*.json", "*.txt", "*.log"]:
            for file_path in output_dir.glob(pattern):
                wandb.save(str(file_path))
        model_artifact = wandb.Artifact(
            name=f"{config['run_name']}-checkpoints",
            type="model",
            metadata={"run_name": config["run_name"], "model_variant": config["model_architecture"]},
        )
        added = False
        for checkpoint_name in ["checkpoint_best_total.pth", "checkpoint_best_ema.pth", "checkpoint_best_regular.pth", "checkpoint.pth"]:
            checkpoint_path = output_dir / checkpoint_name
            if checkpoint_path.exists():
                model_artifact.add_file(str(checkpoint_path), name=checkpoint_name)
                added = True
        if added:
            wandb.run.log_artifact(model_artifact)
    except Exception as exc:
        print(f"Could not log RF-DETR artifacts to W&B: {exc}")


def log_event(name, payload=None):
    payload = payload or {}
    print(f"[{datetime.now().isoformat(timespec='seconds')}] {name}: {json.dumps(payload, default=str)}")
    if wandb is not None and wandb.run is not None:
        try:
            wandb.log({f"event/{name}": 1, "event/payload": json.dumps(payload, default=str)})
        except Exception as exc:
            print(f"Could not log event to W&B: {exc}")


def alert_wandb(title, text, level="INFO"):
    if wandb is None or wandb.run is None:
        print(f"W&B alert skipped: {title} - {text}")
        return
    try:
        wandb.alert(title=title, text=text, level=getattr(wandb.AlertLevel, level, wandb.AlertLevel.INFO))
    except Exception as exc:
        print(f"Could not send W&B alert: {exc}")


def attach_wandb_epoch_logger(model, config):
    if wandb is None or not hasattr(model, "callbacks"):
        return
    state = {"dataset_logged": False}

    def _log_epoch(log_stats):
        if wandb.run is None:
            return
        if not state["dataset_logged"]:
            log_dataset_metadata_to_wandb(config)
            state["dataset_logged"] = True
        payload = {}
        for key, value in (log_stats or {}).items():
            if key == "epoch":
                payload["epoch"] = value
            else:
                flatten_metrics(f"rfdetr/{key}", value, payload)
        if payload:
            wandb.log(payload)

    def _log_batch(callback_dict):
        if wandb.run is None:
            return
        if not state["dataset_logged"]:
            log_dataset_metadata_to_wandb(config)
            state["dataset_logged"] = True
        step = int(callback_dict.get("step", 0))
        if step % 100 == 0:
            wandb.log({"trainer/global_step": step, "epoch": callback_dict.get("epoch", 0)})

    def _log_train_end():
        if wandb.run is None:
            return
        log_artifacts_to_wandb(config, config["output_dir"])

    model.callbacks["on_fit_epoch_end"].append(_log_epoch)
    model.callbacks["on_train_batch_start"].append(_log_batch)
    model.callbacks["on_train_end"].append(_log_train_end)


def train_rfdetr_benchmark_run(config):
    start_time = time.time()
    output_dir = Path(config["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)
    run_config_path = RUN_CONFIG_DIR / f"{config['run_name']}.json"
    run_config_path.write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")

    wandb_enabled = prepare_wandb(config)
    model_class = RFDETR_MODEL_REGISTRY[config["model_architecture"]]["class"]

    train_args = {
        "dataset_dir": config["dataset_dir"],
        "output_dir": str(output_dir),
        "epochs": config["epochs"],
        "batch_size": config["batch_size"],
        "grad_accum_steps": config["grad_accum_steps"],
        "lr": config["lr"],
        "lr_encoder": config["lr_encoder"],
        "lr_drop": config["lr_drop"],
        "resolution": config["resolution"],
        "multi_scale": config["multi_scale"],
        "expanded_scales": config["expanded_scales"],
        "do_random_resize_via_padding": config["do_random_resize_via_padding"],
        "weight_decay": config["weight_decay"],
        "warmup_epochs": config["warmup_epochs"],
        "device": config["device"],
        "num_workers": config["workers"],
        "use_ema": config["use_ema"],
        "gradient_checkpointing": config["gradient_checkpointing"],
        "checkpoint_interval": config["checkpoint_interval"],
        "early_stopping": config["early_stopping"],
        "early_stopping_patience": config["early_stopping_patience"],
        "early_stopping_min_delta": config["early_stopping_min_delta"],
        "class_names": dataset_metadata(config).get("class_names", []),
        "tensorboard": True,
        "wandb": wandb_enabled,
        "project": config["wandb"]["project"],
        "run": config["run_name"],
    }

    row = {
        "model_variant": config["model_architecture"],
        "run_name": config["run_name"],
        "status": "started",
        "output_dir": str(output_dir),
        "wandb_project": config["wandb"]["project"],
    }

    try:
        model = model_class()
        attach_wandb_epoch_logger(model, config)
        log_event("training_start", {"model_variant": config["model_architecture"], "output_dir": str(output_dir)})
        train_result = model.train(**train_args)
        train_seconds = time.time() - start_time
        metrics = extract_train_result_metrics(train_result)
        checkpoints = checkpoint_summary(output_dir)

        config.update({
            "train_seconds": train_seconds,
            "train_args": train_args,
            "checkpoints": checkpoints,
            "final_metrics": metrics,
            "finished_at": datetime.now().isoformat(timespec="seconds"),
        })
        run_config_path.write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")

        if wandb_enabled:
            log_dataset_metadata_to_wandb(config)
            log_artifacts_to_wandb(config, output_dir)
            if wandb is not None and wandb.run is not None:
                wandb.log({**{f"final/{key}": value for key, value in metrics.items()}, "runtime/train_seconds": train_seconds})

        row.update({
            "status": "success",
            "total_training_time_seconds": round(train_seconds, 2),
            "total_training_time_minutes": round(train_seconds / 60, 2),
            "checkpoint_count": len(checkpoints),
            "best_total_checkpoint": str(output_dir / "checkpoint_best_total.pth") if (output_dir / "checkpoint_best_total.pth").exists() else None,
            "best_ema_checkpoint": str(output_dir / "checkpoint_best_ema.pth") if (output_dir / "checkpoint_best_ema.pth").exists() else None,
            "best_regular_checkpoint": str(output_dir / "checkpoint_best_regular.pth") if (output_dir / "checkpoint_best_regular.pth").exists() else None,
            **metrics,
        })
        log_event("training_end", {"model_variant": config["model_architecture"], "train_seconds": round(train_seconds, 2), "status": "success"})
        alert_wandb(f"{config['model_architecture']} benchmark completed", f"runtime={round(train_seconds / 60, 2)} min; status=success", "INFO")
        return row
    except Exception as exc:
        train_seconds = time.time() - start_time
        error_text = repr(exc)
        cuda_context_invalid = "device-side assert" in error_text or "cudaErrorAssert" in error_text
        row.update({"status": "failed", "error": error_text, "total_training_time_seconds": round(train_seconds, 2), "cuda_context_invalid": cuda_context_invalid})
        config.update({"status": "failed", "error": error_text, "cuda_context_invalid": cuda_context_invalid, "train_seconds": train_seconds, "finished_at": datetime.now().isoformat(timespec="seconds")})
        run_config_path.write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")
        log_event("training_failed", {"model_variant": config.get("model_architecture"), "error": error_text, "train_seconds": round(train_seconds, 2), "cuda_context_invalid": cuda_context_invalid})
        alert_wandb(f"{config.get('model_architecture')} benchmark failed", f"status=failure; runtime={round(train_seconds / 60, 2)} min; error={error_text}", "ERROR")
        return row
    finally:
        finish_wandb_run()


## Run Multi-Model Template


In [ ]:
# DO NOT RUN unless training is intended - the full model-scale benchmark
# trains every variant in sequence. Flip to True only for a real training session.
RUN_TRAINING = False

MODEL_RUN_CONFIG_DIR = RUN_CONFIG_DIR
for directory in [BENCHMARK_RESULTS_DIR, MODEL_RUN_CONFIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if importlib.util.find_spec("psutil") is not None:
    try:
        import psutil  # type: ignore
        vm = psutil.virtual_memory()
        HARDWARE_INFO.update({
            "cpu_count_physical": psutil.cpu_count(logical=False),
            "cpu_count_logical": psutil.cpu_count(logical=True),
            "ram_total_gb": round(vm.total / (1024 ** 3), 2),
            "ram_available_gb": round(vm.available / (1024 ** 3), 2),
        })
    except Exception as exc:
        HARDWARE_INFO["psutil_warning"] = str(exc)


def make_model_run_config(base_config, model_variant, experiment_index):
    config = deepcopy(base_config)
    display_name = RFDETR_MODEL_REGISTRY[model_variant]["display_name"]
    config["experiment_type"] = "model-scale-benchmark"
    config["model_architecture"] = model_variant
    config["model_display_name"] = display_name
    config["benchmark_variant"] = model_variant
    experiment_prefix = EXPERIMENT_PREFIX_TEMPLATE.format(index=experiment_index)
    config["experiment_prefix"] = experiment_prefix
    config["experiment_index"] = experiment_index
    config["run_name"] = safe_run_name(f"{experiment_prefix}_{model_variant}_mtsdv1_coco_img640_b32_e100_adamw_lr0p001")
    config["output_dir"] = str(Path(config["project_dir"]) / config["run_name"])
    config["wandb_group"] = EXPERIMENT_FAMILY
    config["tags"] = sorted(set(config.get("tags", []) + [model_variant, display_name, "model-scale-benchmark", "scale-ablation"]))
    config["notes"] = f"Template run for {display_name}: identical shared training settings, only the RF-DETR model class changes."
    config["hardware"] = HARDWARE_INFO
    config["created_at"] = datetime.now().isoformat(timespec="seconds")
    return config


model_run_configs = [
    make_model_run_config(base_training_config, variant, index)
    for index, variant in enumerate(RFDETR_MODEL_VARIANTS, start=1)
]

for config in model_run_configs:
    (MODEL_RUN_CONFIG_DIR / f"{config['run_name']}.json").write_text(json.dumps(config, indent=2, default=str), encoding="utf-8")

metadata = dataset_metadata(base_training_config)
metadata_path = BENCHMARK_RESULTS_DIR / "dataset_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print("Dataset metadata saved to:", metadata_path)
print(json.dumps(metadata, indent=2, default=str))

benchmark_rows = []
if RUN_TRAINING:
    for index, config in enumerate(model_run_configs, start=1):
        print(f"\n=== RF-DETR model-scale benchmark {index}/{len(model_run_configs)}: {config['model_display_name']} ===")
        row = train_rfdetr_benchmark_run(config)
        benchmark_rows.append(row)
        summary_path = BENCHMARK_RESULTS_DIR / "rfdetr_model_scale_summary.csv"
        pd.DataFrame(benchmark_rows).to_csv(summary_path, index=False)
        print("Updated benchmark summary:", summary_path)
        if row.get("cuda_context_invalid"):
            print("CUDA context is invalid after a device-side assert. Restart the kernel before continuing with the next model.")
            break
else:
    print("RUN_TRAINING is False. Set it to True to start the RF-DETR model-scale benchmark.")

summary_df = pd.DataFrame(benchmark_rows)
summary_csv_path = BENCHMARK_RESULTS_DIR / "rfdetr_model_scale_summary.csv"
summary_json_path = BENCHMARK_RESULTS_DIR / "rfdetr_model_scale_summary.json"
summary_df.to_csv(summary_csv_path, index=False)
summary_json_path.write_text(json.dumps(benchmark_rows, indent=2, default=str), encoding="utf-8")
print("Benchmark CSV saved to:", summary_csv_path)
print("Benchmark JSON saved to:", summary_json_path)
summary_df
